# DACON 딥보이스 탐지 — ASVspoof2019 LA 20K + AASIST Starter

이 노트북은 다음을 한 번에 수행합니다.

1. Google Drive 마운트
2. Drive의 **ASVspoof2019 LA** 데이터 자동 탐색
3. 공식 protocol에서 **정확히 20,000개**를 stratified sampling
4. 20,000개 내부에서 Train/Validation = 80/20 분할
5. 선택한 FLAC을 Colab local disk에 캐시(권장)
6. 공식 AASIST 구조를 사용한 baseline 학습
7. Validation Accuracy / ROC-AUC / EER 평가
8. Best checkpoint 저장
9. DACON 코드 제출 규격에 맞는 `script.py`, `model/`, `requirements.txt` 생성
10. 최상위 폴더가 없는 **submit.zip** 생성 및 구조 검증

## 중요한 한계

ASVspoof2019 LA는 **음성(speech) deepfake / spoof** 데이터입니다.  
DACON 대회는 다음 5개 확률을 요구합니다.

- `FILE_FAKE_PROB`
- `VOICE_FAKE_PROB`
- `MUSIC_FAKE_PROB`
- `VOICE_PRESENT_PROB`
- `MUSIC_PRESENT_PROB`

따라서 이 노트북의 제출 파일은 **Voice/AASIST starter submission**입니다.
ASVspoof2019 LA만으로는 음악 fake와 음악/음성 presence를 제대로 학습할 수 없으므로,
starter `script.py`에서는 음악/presence 출력에 중립값을 사용합니다.

**고득점 단계에서는 FakeMusicCaps / CtrSVDD / 혼합 오디오 등의 추가 학습 데이터와
WPT-XLSR-AASIST, RawBoost/Codec, multi-task 5-head 모델을 추가해야 합니다.**

또한 ASVspoof2019 **PA spoof는 replay attack**이므로, DACON의 "AI 생성 여부"와 의미가 다릅니다.
이 starter에서는 PA를 FAKE 학습 데이터로 합치지 않고 LA만 사용합니다.

## DACON 평가/제출 규격 메모

리더보드:

- `Score = 0.9 * ADS + 0.1 * CPS`
- `ADS = 0.5*(1-File EER) + 0.2*(1-Voice EER) + 0.3*(1-Music EER)`
- `CPS = 0.5*Voice Presence AUC + 0.5*Music Presence AUC`
- DACON에서는 **FAKE = positive class(1)**
- EER/AUC 기반이므로 hard label보다 **연속 score/probability의 순위 품질**이 중요합니다.

코드 제출:

```text
submit.zip
├── model/
├── script.py
└── requirements.txt
```

평가 서버는 `output/submission.csv`를 요구합니다.
평가 페이지의 테스트 입력 폴더 표기가 `data/` / `open/`으로 혼재되어 있어
이 노트북이 생성하는 `script.py`는 두 경로를 모두 탐색합니다.

평가 서버 기본 패키지는 requirements에 다시 강제 설치하지 않는 방향으로 구성합니다.

In [ ]:
# ============================================================
# CELL 1. GPU / Runtime 확인
# ============================================================

!nvidia-smi

import sys
import torch

print("Python :", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA   :", torch.version.cuda)
print("GPU    :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

assert torch.cuda.is_available(), "Colab 런타임을 GPU(T4)로 변경하세요."

In [ ]:
# ============================================================
# CELL 2. 최소 의존성 확인 + AASIST 공식 코드 clone
# 핵심 numpy/torch/sklearn은 Colab 기본 버전을 그대로 사용합니다.
# ============================================================

import importlib.util
import subprocess
import sys
from pathlib import Path

def ensure_package(pkg, pip_name=None):
    if importlib.util.find_spec(pkg) is None:
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", pip_name or pkg
        ])

ensure_package("soundfile", "soundfile")
ensure_package("sklearn", "scikit-learn")

AASIST_ROOT = Path("/content/aasist")
if not AASIST_ROOT.exists():
    subprocess.check_call([
        "git", "clone", "-q",
        "https://github.com/clovaai/aasist.git",
        str(AASIST_ROOT)
    ])

print("AASIST:", AASIST_ROOT)

In [ ]:
# ============================================================
# CELL 3. Imports + Seed
# ============================================================

import os
import sys
import gc
import json
import math
import random
import shutil
import zipfile
import subprocess
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import soundfile as sf
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    roc_auc_score,
    roc_curve,
    confusion_matrix,
    classification_report,
)

from tqdm.auto import tqdm

SEED = 42

def seed_everything(seed=42):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)

In [ ]:
# ============================================================
# CELL 4. Google Drive Mount
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# ============================================================
# CELL 5. ★ 사용자 설정 ★
# 이 셀의 DRIVE_ASVSPOOF_ROOT만 본인 Drive 경로에 맞게 수정하세요.
# ============================================================

# 예시:
# /content/drive/MyDrive/ASVspoof2019
# /content/drive/MyDrive/Dacon/DeepVoice/LA
# /content/drive/MyDrive/Datasets/ASVspoof2019

DRIVE_ASVSPOOF_ROOT = Path(
    "/content/drive/MyDrive/ASVspoof2019"
)

# 프로젝트 산출물 저장 위치
DRIVE_PROJECT_DIR = Path(
    "/content/drive/MyDrive/Dacon/DeepVoice_AASIST_20K"
)

DRIVE_PROJECT_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_CHECKPOINT_DIR = DRIVE_PROJECT_DIR / "checkpoints"
DRIVE_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# 정확히 20,000개 사용 (train + validation 합계)
N_TOTAL = 20_000
VALID_RATIO = 0.20

# Drive의 작은 파일 I/O를 피하기 위해 선택한 20K만 /content에 복사
CACHE_SELECTED_TO_LOCAL = True
LOCAL_CACHE_DIR = Path("/content/asvspoof2019_la_20k")

# T4 안정성 우선
BATCH_SIZE = 8
NUM_WORKERS = 0

# Starter baseline
EPOCHS = 5
LR = 1e-4
WEIGHT_DECAY = 1e-4

# 공식 AASIST input length
MAX_LEN = 64600

# 첫 실행 안정성 우선. 잘 돌아가면 True로 바꿔도 됩니다.
USE_AMP = False

print("DRIVE_ASVSPOOF_ROOT:", DRIVE_ASVSPOOF_ROOT)
print("DRIVE_PROJECT_DIR  :", DRIVE_PROJECT_DIR)

In [ ]:
# ============================================================
# CELL 6. ASVspoof2019 LA 폴더 / protocol 자동 탐색
# ============================================================

assert DRIVE_ASVSPOOF_ROOT.exists(), (
    f"Drive 경로가 없습니다: {DRIVE_ASVSPOOF_ROOT}\n"
    "CELL 5의 DRIVE_ASVSPOOF_ROOT를 수정하세요."
)

train_candidates = list(
    DRIVE_ASVSPOOF_ROOT.rglob("ASVspoof2019_LA_train")
)

assert train_candidates, (
    "ASVspoof2019_LA_train 폴더를 찾지 못했습니다.\n"
    "DRIVE_ASVSPOOF_ROOT가 LA 폴더를 포함하는 상위 경로인지 확인하세요."
)

TRAIN_DIR_DRIVE = train_candidates[0]
LA_ROOT_DRIVE = TRAIN_DIR_DRIVE.parent

protocol_dir_candidates = list(
    LA_ROOT_DRIVE.rglob("ASVspoof2019_LA_cm_protocols")
)
assert protocol_dir_candidates, "ASVspoof2019_LA_cm_protocols를 찾지 못했습니다."

PROTOCOL_DIR = protocol_dir_candidates[0]
TRAIN_PROTOCOL = PROTOCOL_DIR / "ASVspoof2019.LA.cm.train.trn.txt"

assert TRAIN_PROTOCOL.exists(), f"Train protocol 없음: {TRAIN_PROTOCOL}"
assert (TRAIN_DIR_DRIVE / "flac").exists(), "LA train/flac 폴더가 없습니다."

print("LA_ROOT_DRIVE :", LA_ROOT_DRIVE)
print("TRAIN_DIR     :", TRAIN_DIR_DRIVE)
print("TRAIN_PROTOCOL:", TRAIN_PROTOCOL)

In [ ]:
# ============================================================
# CELL 7. Protocol → DataFrame
# 공식 AASIST convention:
#   0 = spoof (FAKE)
#   1 = bonafide (REAL)
# DACON 제출에서는 fake probability가 필요하므로 추론 시 class 0 prob를 사용합니다.
# ============================================================

def read_la_protocol(protocol_path, train_dir):
    rows = []
    with open(protocol_path, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) != 5:
                continue

            speaker_id, utt_id, _, attack_id, label_text = parts
            aasist_label = 1 if label_text == "bonafide" else 0
            src_path = train_dir / "flac" / f"{utt_id}.flac"

            rows.append({
                "speaker_id": speaker_id,
                "utt_id": utt_id,
                "attack_id": attack_id,
                "label_text": label_text,
                "label": aasist_label,
                "fake_label": 1 - aasist_label,
                "src_path": str(src_path),
            })
    return pd.DataFrame(rows)

full_df = read_la_protocol(TRAIN_PROTOCOL, TRAIN_DIR_DRIVE)

print("전체 protocol:", len(full_df))
display(full_df.head())

print("\nClass counts")
display(full_df["label_text"].value_counts())

In [ ]:
# ============================================================
# CELL 8. 정확히 20,000개 Stratified Sampling + 80/20 split
# ============================================================

assert len(full_df) >= N_TOTAL, f"원본 데이터가 {N_TOTAL}개보다 적습니다."

sampled_df, _ = train_test_split(
    full_df,
    train_size=N_TOTAL,
    stratify=full_df["label"],
    random_state=SEED,
)

train_df, val_df = train_test_split(
    sampled_df,
    test_size=VALID_RATIO,
    stratify=sampled_df["label"],
    random_state=SEED,
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

sampled_manifest = pd.concat([
    train_df.assign(split="train"),
    val_df.assign(split="val"),
], ignore_index=True)

manifest_path = DRIVE_PROJECT_DIR / "asvspoof_la_sample_20000.csv"
sampled_manifest.to_csv(manifest_path, index=False, encoding="utf-8")

print("TOTAL:", len(sampled_manifest))
print("TRAIN:", len(train_df))
print("VAL  :", len(val_df))
print("\nTrain labels:")
print(train_df["label_text"].value_counts())
print("\nVal labels:")
print(val_df["label_text"].value_counts())
print("\nManifest:", manifest_path)

In [ ]:
# ============================================================
# CELL 9. 선택한 20K FLAC을 Colab local disk에 캐시 (권장)
# 이미 복사된 파일은 skip합니다.
# ============================================================

if CACHE_SELECTED_TO_LOCAL:
    CACHE_FLAC_DIR = LOCAL_CACHE_DIR / "flac"
    CACHE_FLAC_DIR.mkdir(parents=True, exist_ok=True)

    unique_paths = sampled_manifest[["utt_id", "src_path"]].drop_duplicates()

    for row in tqdm(
        unique_paths.itertuples(index=False),
        total=len(unique_paths),
        desc="Drive -> Colab cache"
    ):
        src = Path(row.src_path)
        dst = CACHE_FLAC_DIR / f"{row.utt_id}.flac"

        if not dst.exists():
            if not src.exists():
                raise FileNotFoundError(src)
            shutil.copy2(src, dst)

    def choose_path(utt_id, src_path):
        return str(CACHE_FLAC_DIR / f"{utt_id}.flac")

else:
    def choose_path(utt_id, src_path):
        return src_path

train_df["path"] = [
    choose_path(u, p)
    for u, p in zip(train_df["utt_id"], train_df["src_path"])
]
val_df["path"] = [
    choose_path(u, p)
    for u, p in zip(val_df["utt_id"], val_df["src_path"])
]

print("Example:", train_df.loc[0, "path"])
print("Exists :", Path(train_df.loc[0, "path"]).exists())

## EDA

AASIST는 raw waveform을 직접 사용하므로 과도한 denoise / enhancement를 하지 않습니다.

전처리:

- FLAC load
- stereo이면 mono average
- ASVspoof2019 LA는 일반적으로 16 kHz
- 길면 64,600 samples random crop (train)
- validation은 deterministic center crop
- 짧으면 반복하여 64,600 samples로 맞춤

In [ ]:
# ============================================================
# CELL 10. 간단한 EDA
# ============================================================

eda_df = sampled_manifest.sample(
    n=min(300, len(sampled_manifest)),
    random_state=SEED,
).copy()

durations = []
sample_rates = []

for p in tqdm(eda_df["src_path"], desc="EDA audio info"):
    info = sf.info(p)
    durations.append(info.duration)
    sample_rates.append(info.samplerate)

eda_df["duration_sec"] = durations
eda_df["sample_rate"] = sample_rates

display(eda_df[["duration_sec", "sample_rate", "label_text"]].describe(include="all"))

plt.figure(figsize=(8, 4))
plt.hist(eda_df["duration_sec"], bins=30)
plt.xlabel("Duration (sec)")
plt.ylabel("Count")
plt.title("ASVspoof2019 LA sampled duration")
plt.show()

In [ ]:
# ============================================================
# CELL 11. Dataset
# ============================================================

def pad_or_crop(x, max_len=64600, train=True):
    x = np.asarray(x, dtype=np.float32)

    if len(x) == 0:
        return np.zeros(max_len, dtype=np.float32)

    if len(x) >= max_len:
        if train:
            start = np.random.randint(0, len(x) - max_len + 1)
        else:
            start = max((len(x) - max_len) // 2, 0)
        return x[start:start + max_len]

    repeat = (max_len // len(x)) + 1
    return np.tile(x, repeat)[:max_len].astype(np.float32)


class ASVspoofLADataset(Dataset):
    def __init__(self, df, train=True, max_len=64600):
        self.df = df.reset_index(drop=True)
        self.train = train
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        audio, sr = sf.read(
            row["path"],
            dtype="float32",
            always_2d=False,
        )

        if audio.ndim > 1:
            audio = audio.mean(axis=1)

        if sr != 16000:
            raise ValueError(
                f"Expected 16kHz ASVspoof audio, got {sr}: {row['path']}"
            )

        audio = pad_or_crop(
            audio,
            max_len=self.max_len,
            train=self.train,
        )

        return (
            torch.from_numpy(audio),
            torch.tensor(int(row["label"]), dtype=torch.long),
            row["utt_id"],
        )


train_dataset = ASVspoofLADataset(train_df, train=True, max_len=MAX_LEN)
val_dataset = ASVspoofLADataset(val_df, train=False, max_len=MAX_LEN)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=False,
    drop_last=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=False,
    drop_last=False,
)

x, y, ids = next(iter(train_loader))
print("Audio:", x.shape)
print("Label:", y.shape)
print("IDs  :", ids[:3])

assert x.shape[1] == MAX_LEN

In [ ]:
# ============================================================
# CELL 12. 공식 AASIST Config + Model
# ============================================================

if str(AASIST_ROOT) not in sys.path:
    sys.path.insert(0, str(AASIST_ROOT))

from models.AASIST import Model as AASIST

CONFIG_PATH = AASIST_ROOT / "config" / "AASIST.conf"
with open(CONFIG_PATH, "r") as f:
    official_config = json.load(f)

MODEL_CONFIG = official_config["model_config"]

model = AASIST(MODEL_CONFIG).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters())
print("Parameters:", f"{n_params:,}")

# Forward sanity check
model.eval()
test_x = x[:2].to(DEVICE)
with torch.no_grad():
    hidden, logits = model(test_x, Freq_aug=False)

print("Input :", test_x.shape)
print("Hidden:", hidden.shape)
print("Logits:", logits.shape)

assert logits.shape == (2, 2)

In [ ]:
# ============================================================
# CELL 13. DACON/ASVspoof Metric 함수
# AASIST label: 0=spoof, 1=bonafide
# DACON positive: FAKE=1
# 따라서 fake_true = 1 - aasist_label
# ============================================================

def calculate_eer(y_true_fake, fake_scores):
    fpr, tpr, thresholds = roc_curve(
        y_true_fake,
        fake_scores,
        pos_label=1,
        drop_intermediate=False,
    )
    fnr = 1.0 - tpr
    idx = np.argmin(np.abs(fpr - fnr))
    eer = (fpr[idx] + fnr[idx]) / 2.0
    return float(eer), float(thresholds[idx])


def dacon_full_score(y_true, y_pred):
    # 향후 Voice/Music/Presence 라벨 데이터에서 사용
    file_eer, _ = calculate_eer(
        y_true["FILE_FAKE_PROB"],
        y_pred["FILE_FAKE_PROB"],
    )

    voice_mask = np.asarray(y_true["VOICE_PRESENT_PROB"]) == 1
    music_mask = np.asarray(y_true["MUSIC_PRESENT_PROB"]) == 1

    voice_eer, _ = calculate_eer(
        np.asarray(y_true["VOICE_FAKE_PROB"])[voice_mask],
        np.asarray(y_pred["VOICE_FAKE_PROB"])[voice_mask],
    )

    music_eer, _ = calculate_eer(
        np.asarray(y_true["MUSIC_FAKE_PROB"])[music_mask],
        np.asarray(y_pred["MUSIC_FAKE_PROB"])[music_mask],
    )

    voice_auc = roc_auc_score(
        y_true["VOICE_PRESENT_PROB"],
        y_pred["VOICE_PRESENT_PROB"],
    )
    music_auc = roc_auc_score(
        y_true["MUSIC_PRESENT_PROB"],
        y_pred["MUSIC_PRESENT_PROB"],
    )

    ads = (
        0.5 * (1 - file_eer)
        + 0.2 * (1 - voice_eer)
        + 0.3 * (1 - music_eer)
    )
    cps = 0.5 * voice_auc + 0.5 * music_auc
    score = 0.9 * ads + 0.1 * cps

    return {
        "score": score,
        "ADS": ads,
        "CPS": cps,
        "file_eer": file_eer,
        "voice_eer": voice_eer,
        "music_eer": music_eer,
        "voice_auc": voice_auc,
        "music_auc": music_auc,
    }

In [ ]:
# ============================================================
# CELL 14. Loss / Optimizer / Scheduler
# ============================================================

class_weights = torch.tensor(
    [0.1, 0.9],
    dtype=torch.float32,
    device=DEVICE,
)

criterion = nn.CrossEntropyLoss(weight=class_weights)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LR,
    betas=(0.9, 0.999),
    weight_decay=WEIGHT_DECAY,
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=EPOCHS,
    eta_min=5e-6,
)

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=(USE_AMP and DEVICE.type == "cuda")
)

print("Loss/Optimizer ready")

In [ ]:
# ============================================================
# CELL 15. Train / Validation
# Best model = Validation EER 최소
# ============================================================

def train_one_epoch(model, loader):
    model.train()

    losses = []
    y_true = []
    y_pred = []

    pbar = tqdm(loader, desc="TRAIN", leave=False)

    for audio, labels, _ in pbar:
        audio = audio.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(USE_AMP and DEVICE.type == "cuda")
        ):
            _, logits = model(audio, Freq_aug=False)
            loss = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        pred = logits.argmax(dim=1)

        losses.append(loss.item())
        y_true.extend(labels.detach().cpu().numpy())
        y_pred.extend(pred.detach().cpu().numpy())

        pbar.set_postfix(loss=f"{loss.item():.4f}")

    return {
        "loss": float(np.mean(losses)),
        "accuracy": accuracy_score(y_true, y_pred),
    }


@torch.inference_mode()
def validate(model, loader):
    model.eval()

    losses = []
    aasist_true = []
    aasist_pred = []
    fake_scores = []
    utt_ids = []

    for audio, labels, ids in tqdm(loader, desc="VAL", leave=False):
        audio = audio.to(DEVICE)
        labels = labels.to(DEVICE)

        _, logits = model(audio, Freq_aug=False)
        loss = criterion(logits, labels)

        probs = torch.softmax(logits.float(), dim=1)

        # class 0 = spoof/fake
        batch_fake_scores = probs[:, 0]
        preds = logits.argmax(dim=1)

        losses.append(loss.item())
        aasist_true.extend(labels.cpu().numpy())
        aasist_pred.extend(preds.cpu().numpy())
        fake_scores.extend(batch_fake_scores.cpu().numpy())
        utt_ids.extend(ids)

    aasist_true = np.asarray(aasist_true)
    aasist_pred = np.asarray(aasist_pred)
    fake_scores = np.asarray(fake_scores)

    fake_true = 1 - aasist_true

    eer, eer_threshold = calculate_eer(fake_true, fake_scores)

    return {
        "loss": float(np.mean(losses)),
        "accuracy": accuracy_score(aasist_true, aasist_pred),
        "balanced_accuracy": balanced_accuracy_score(aasist_true, aasist_pred),
        "f1_bonafide": f1_score(aasist_true, aasist_pred, pos_label=1),
        "fake_auc": roc_auc_score(fake_true, fake_scores),
        "eer": eer,
        "eer_threshold": eer_threshold,
        "labels": aasist_true,
        "preds": aasist_pred,
        "fake_scores": fake_scores,
        "utt_ids": utt_ids,
    }

In [ ]:
# ============================================================
# CELL 16. 학습
# ============================================================

BEST_CKPT = DRIVE_CHECKPOINT_DIR / "aasist_la20k_best.pt"
HISTORY_CSV = DRIVE_PROJECT_DIR / "aasist_la20k_history.csv"

best_eer = float("inf")
history = []

for epoch in range(1, EPOCHS + 1):
    print(f"\n{'='*70}")
    print(f"EPOCH {epoch}/{EPOCHS}")
    print(f"{'='*70}")

    train_result = train_one_epoch(model, train_loader)
    val_result = validate(model, val_loader)

    scheduler.step()

    row = {
        "epoch": epoch,
        "train_loss": train_result["loss"],
        "train_accuracy": train_result["accuracy"],
        "val_loss": val_result["loss"],
        "val_accuracy": val_result["accuracy"],
        "val_balanced_accuracy": val_result["balanced_accuracy"],
        "val_fake_auc": val_result["fake_auc"],
        "val_eer": val_result["eer"],
    }
    history.append(row)
    pd.DataFrame(history).to_csv(HISTORY_CSV, index=False)

    print(
        f"train_loss={row['train_loss']:.4f} "
        f"train_acc={row['train_accuracy']:.4f}"
    )
    print(
        f"val_acc={row['val_accuracy']:.4f} "
        f"balanced_acc={row['val_balanced_accuracy']:.4f} "
        f"fake_auc={row['val_fake_auc']:.4f} "
        f"EER={row['val_eer']*100:.3f}%"
    )

    if val_result["eer"] < best_eer:
        best_eer = val_result["eer"]

        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "dev_eer": float(best_eer),
            "model_config": MODEL_CONFIG,
        }, BEST_CKPT)

        print("★ BEST SAVED:", BEST_CKPT)

    gc.collect()
    torch.cuda.empty_cache()

print("\nBest EER:", f"{best_eer*100:.3f}%")

In [ ]:
# ============================================================
# CELL 17. Best model 최종 평가
# ============================================================

checkpoint = torch.load(
    BEST_CKPT,
    map_location=DEVICE,
    weights_only=False,
)

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

final_val = validate(model, val_loader)

print("Best epoch        :", checkpoint["epoch"])
print("Validation Acc    :", f"{final_val['accuracy']*100:.3f}%")
print("Balanced Acc      :", f"{final_val['balanced_accuracy']*100:.3f}%")
print("Fake ROC-AUC      :", f"{final_val['fake_auc']:.6f}")
print("EER               :", f"{final_val['eer']*100:.3f}%")

print("\nClassification Report (AASIST label: 0=spoof, 1=bonafide)")
print(classification_report(
    final_val["labels"],
    final_val["preds"],
    target_names=["SPOOF/FAKE", "BONAFIDE/REAL"],
    digits=4,
))

# DACON 제출 ZIP 생성

이 starter 모델은 ASVspoof2019 LA의 **voice fake detector**입니다.

제출 변환:

- `VOICE_FAKE_PROB` = AASIST spoof(class 0) probability
- `FILE_FAKE_PROB` = voice fake probability
- `MUSIC_FAKE_PROB` = 0.5 (중립 baseline)
- `VOICE_PRESENT_PROB` = 0.5 (중립 baseline)
- `MUSIC_PRESENT_PROB` = 0.5 (중립 baseline)

긴 테스트 파일(최대 1분)은 64,600 sample 단위로 segmentation하고
상위 segment score의 평균(top-k mean)을 사용합니다.
이는 하나의 파일 내부 segment 정보를 종합하는 방식입니다.

**이 submit.zip은 제출 형식/추론 파이프라인 검증용 starter입니다.**
높은 leaderboard 점수를 목표로 할 경우 음악/혼합 데이터와 5-head multi-task 모델이 필수입니다.

In [ ]:
# ============================================================
# CELL 18. submit.zip 작업 폴더 생성
# ============================================================

SUBMIT_BUILD_DIR = Path("/content/dacon_submit_build")
SUBMIT_MODEL_DIR = SUBMIT_BUILD_DIR / "model"

if SUBMIT_BUILD_DIR.exists():
    shutil.rmtree(SUBMIT_BUILD_DIR)

SUBMIT_MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("Build dir:", SUBMIT_BUILD_DIR)

In [ ]:
# ============================================================
# CELL 19. 학습한 weights + AASIST 소스 + config 패키징
# ============================================================

ckpt = torch.load(
    BEST_CKPT,
    map_location="cpu",
    weights_only=False,
)

torch.save(
    ckpt["model_state_dict"],
    SUBMIT_MODEL_DIR / "aasist_weights.pt",
)

with open(
    SUBMIT_MODEL_DIR / "model_config.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(MODEL_CONFIG, f, ensure_ascii=False, indent=2)

shutil.copy2(
    AASIST_ROOT / "models" / "AASIST.py",
    SUBMIT_MODEL_DIR / "aasist_model.py",
)

print("Packaged model files:")
for p in sorted(SUBMIT_MODEL_DIR.iterdir()):
    print(" -", p.name, f"({p.stat().st_size/1024/1024:.2f} MB)")

In [ ]:
# ============================================================
# CELL 20. DACON script.py 생성
# data/ 와 open/ 입력 경로를 모두 지원
# 긴 파일은 segment -> top-k mean
# ============================================================

script_code = 'import json\nimport math\nimport sys\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport torch\nimport torchaudio\nimport librosa\n\nROOT = Path(__file__).resolve().parent\nMODEL_DIR = ROOT / "model"\nOUTPUT_DIR = ROOT / "output"\nOUTPUT_DIR.mkdir(parents=True, exist_ok=True)\n\nsys.path.insert(0, str(MODEL_DIR))\nfrom aasist_model import Model as AASIST\n\nPRED_COLS = [\n    "FILE_FAKE_PROB",\n    "VOICE_FAKE_PROB",\n    "MUSIC_FAKE_PROB",\n    "VOICE_PRESENT_PROB",\n    "MUSIC_PRESENT_PROB",\n]\n\nTARGET_SR = 16000\nSEG_LEN = 64600\nHOP_LEN = SEG_LEN // 2\nMAX_SEGMENTS = 12\nINFER_BATCH = 16\nTOP_RATIO = 0.30\n\nDEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")\nif torch.cuda.is_available():\n    torch.backends.cudnn.benchmark = True\n\n\ndef find_input_root():\n    candidates = [ROOT / "data", ROOT / "open"]\n    for p in candidates:\n        if p.exists():\n            return p\n    raise FileNotFoundError(\n        "Evaluation input directory not found. Checked: data/, open/"\n    )\n\n\ndef find_sample_submission(input_root):\n    candidates = list(input_root.rglob("sample_submission.csv"))\n    return candidates[0] if candidates else None\n\n\ndef list_audio_files(input_root):\n    exts = {".wav", ".flac", ".mp3", ".m4a", ".ogg", ".aac"}\n    return sorted([\n        p for p in input_root.rglob("*")\n        if p.is_file() and p.suffix.lower() in exts\n    ])\n\n\ndef build_audio_index(audio_files):\n    idx = {}\n    for p in audio_files:\n        idx.setdefault(p.name, p)\n        idx.setdefault(p.stem, p)\n    return idx\n\n\ndef load_audio(path):\n    try:\n        wav, sr = torchaudio.load(str(path))\n        wav = wav.float()\n        if wav.ndim == 2:\n            wav = wav.mean(dim=0)\n        else:\n            wav = wav.reshape(-1)\n\n        if sr != TARGET_SR:\n            wav = torchaudio.functional.resample(\n                wav,\n                orig_freq=sr,\n                new_freq=TARGET_SR,\n            )\n        return wav.cpu()\n\n    except Exception:\n        y, _ = librosa.load(\n            str(path),\n            sr=TARGET_SR,\n            mono=True,\n        )\n        return torch.tensor(y, dtype=torch.float32)\n\n\ndef repeat_pad(x, length):\n    if x.numel() == 0:\n        return torch.zeros(length, dtype=torch.float32)\n    if x.numel() >= length:\n        return x[:length]\n    reps = math.ceil(length / x.numel())\n    return x.repeat(reps)[:length]\n\n\ndef make_segments(wav):\n    n = wav.numel()\n\n    if n <= SEG_LEN:\n        return repeat_pad(wav, SEG_LEN).unsqueeze(0)\n\n    starts = list(range(0, n - SEG_LEN + 1, HOP_LEN))\n    last = n - SEG_LEN\n    if starts[-1] != last:\n        starts.append(last)\n\n    if len(starts) > MAX_SEGMENTS:\n        pick = np.linspace(\n            0,\n            len(starts) - 1,\n            MAX_SEGMENTS,\n        ).round().astype(int)\n        starts = [starts[i] for i in pick]\n\n    return torch.stack([\n        wav[s:s + SEG_LEN]\n        for s in starts\n    ])\n\n\ndef topk_mean(scores, top_ratio=0.30):\n    scores = np.asarray(scores, dtype=np.float64)\n    k = max(1, int(math.ceil(len(scores) * top_ratio)))\n    return float(np.sort(scores)[-k:].mean())\n\n\ndef load_model():\n    with open(MODEL_DIR / "model_config.json", "r") as f:\n        config = json.load(f)\n\n    model = AASIST(config).to(DEVICE)\n    state_dict = torch.load(\n        MODEL_DIR / "aasist_weights.pt",\n        map_location=DEVICE,\n        weights_only=True,\n    )\n    model.load_state_dict(state_dict)\n    model.eval()\n    return model\n\n\n@torch.inference_mode()\ndef predict_voice_fake(model, audio_path):\n    wav = load_audio(audio_path)\n    segments = make_segments(wav)\n\n    all_scores = []\n\n    for start in range(0, len(segments), INFER_BATCH):\n        batch = segments[start:start + INFER_BATCH].to(\n            DEVICE,\n            non_blocking=True,\n        )\n\n        _, logits = model(batch, Freq_aug=False)\n\n        # Official AASIST convention:\n        # class 0 = spoof/fake, class 1 = bonafide/real\n        fake_prob = torch.softmax(\n            logits.float(),\n            dim=1,\n        )[:, 0]\n\n        all_scores.extend(\n            fake_prob.detach().cpu().numpy().tolist()\n        )\n\n    return topk_mean(all_scores, TOP_RATIO)\n\n\ndef main():\n    input_root = find_input_root()\n    audio_files = list_audio_files(input_root)\n\n    if not audio_files:\n        raise RuntimeError(f"No audio files found under {input_root}")\n\n    audio_index = build_audio_index(audio_files)\n    sample_path = find_sample_submission(input_root)\n    model = load_model()\n\n    if sample_path is not None:\n        submission = pd.read_csv(sample_path)\n\n        id_candidates = [\n            c for c in submission.columns\n            if c not in PRED_COLS\n        ]\n        id_col = id_candidates[0] if id_candidates else submission.columns[0]\n\n        resolved = []\n        for value in submission[id_col].astype(str):\n            p = audio_index.get(value)\n            if p is None:\n                p = audio_index.get(Path(value).name)\n            if p is None:\n                p = audio_index.get(Path(value).stem)\n            resolved.append(p)\n\n        if any(p is None for p in resolved):\n            if len(submission) == len(audio_files):\n                resolved = audio_files\n            else:\n                missing = sum(p is None for p in resolved)\n                raise FileNotFoundError(\n                    f"Could not match {missing} submission IDs to audio files."\n                )\n    else:\n        submission = pd.DataFrame({\n            "ID": [p.stem for p in audio_files]\n        })\n        resolved = audio_files\n\n    file_scores = []\n\n    for i, path in enumerate(resolved, 1):\n        score = predict_voice_fake(model, path)\n        file_scores.append(score)\n\n        if i % 50 == 0:\n            print(f"[{i}/{len(resolved)}]")\n\n    file_scores = np.asarray(file_scores, dtype=np.float64)\n\n    submission["FILE_FAKE_PROB"] = file_scores\n    submission["VOICE_FAKE_PROB"] = file_scores\n\n    # Neutral outputs: valid format, but not leaderboard-optimal.\n    submission["MUSIC_FAKE_PROB"] = 0.5\n    submission["VOICE_PRESENT_PROB"] = 0.5\n    submission["MUSIC_PRESENT_PROB"] = 0.5\n\n    for c in PRED_COLS:\n        submission[c] = submission[c].clip(0.0, 1.0)\n\n    out_path = OUTPUT_DIR / "submission.csv"\n    submission.to_csv(\n        out_path,\n        index=False,\n        encoding="utf-8",\n    )\n\n    print("Saved:", out_path)\n    print("Rows :", len(submission))\n\n\nif __name__ == "__main__":\n    main()\n'

SCRIPT_PATH = SUBMIT_BUILD_DIR / "script.py"
SCRIPT_PATH.write_text(
    script_code,
    encoding="utf-8",
)

print("script.py:", SCRIPT_PATH)

In [ ]:
# ============================================================
# CELL 21. requirements.txt 생성
# DACON 기본 설치 패키지는 재설치하지 않습니다.
# ============================================================

requirements_lines = [
    "# No additional pip packages are required.",
    "# This submission intentionally uses DACON preinstalled packages:",
    "# torch==2.7.1+cu128",
    "# torchaudio==2.7.1+cu128",
    "# pandas==2.0.3",
    "# numpy==1.26.4",
    "# librosa==0.10.2.post1",
    "#",
    "# AASIST source is bundled under model/aasist_model.py.",
    "# No GitHub/network download is required during inference.",
]

REQ_PATH = SUBMIT_BUILD_DIR / "requirements.txt"
REQ_PATH.write_text(
    "\n".join(requirements_lines) + "\n",
    encoding="utf-8",
)

print(REQ_PATH.read_text())

In [ ]:
# ============================================================
# CELL 22. submit.zip 생성
# ZIP 내부 최상위는 model/, script.py, requirements.txt
# ============================================================

SUBMIT_ZIP = DRIVE_PROJECT_DIR / "submit.zip"

if SUBMIT_ZIP.exists():
    SUBMIT_ZIP.unlink()

with zipfile.ZipFile(
    SUBMIT_ZIP,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as zf:
    for p in SUBMIT_BUILD_DIR.rglob("*"):
        if p.is_file():
            zf.write(
                p,
                arcname=str(p.relative_to(SUBMIT_BUILD_DIR))
            )

print("Created:", SUBMIT_ZIP)
print("Size   :", f"{SUBMIT_ZIP.stat().st_size/1024/1024:.2f} MB")

In [ ]:
# ============================================================
# CELL 23. ZIP 구조 검증
# ============================================================

with zipfile.ZipFile(SUBMIT_ZIP, "r") as zf:
    names = zf.namelist()

print("\n".join(names))

top_level = sorted({
    n.split("/")[0]
    for n in names
})

print("\nTop level:", top_level)

required = {"model", "script.py", "requirements.txt"}
assert required.issubset(set(top_level)), (
    f"필수 구조가 없습니다. 현재: {top_level}"
)

assert "submit" not in top_level, (
    "ZIP 안에 submit/ 같은 추가 최상위 폴더가 들어가면 안 됩니다."
)

assert SUBMIT_ZIP.stat().st_size < 10 * 1024**3, "10GB 제한 초과"

print("✅ submit.zip 구조/용량 1차 검증 통과")

In [ ]:
# ============================================================
# CELL 24. OPTIONAL: 제출 script 로컬 dry-run
# ============================================================

RUN_DRY_RUN = False

if RUN_DRY_RUN:
    TEST_ROOT = Path("/content/submit_dry_run")

    if TEST_ROOT.exists():
        shutil.rmtree(TEST_ROOT)

    TEST_ROOT.mkdir(parents=True)

    with zipfile.ZipFile(SUBMIT_ZIP, "r") as zf:
        zf.extractall(TEST_ROOT)

    test_data = TEST_ROOT / "data" / "test"
    test_data.mkdir(parents=True)

    dry_df = val_df.head(3).copy()

    for row in dry_df.itertuples(index=False):
        shutil.copy2(
            row.path,
            test_data / f"{row.utt_id}.flac",
        )

    sample = pd.DataFrame({
        "ID": dry_df["utt_id"].tolist(),
        "FILE_FAKE_PROB": 0.0,
        "VOICE_FAKE_PROB": 0.0,
        "MUSIC_FAKE_PROB": 0.0,
        "VOICE_PRESENT_PROB": 0.0,
        "MUSIC_PRESENT_PROB": 0.0,
    })
    sample.to_csv(
        TEST_ROOT / "data" / "sample_submission.csv",
        index=False,
    )

    result = subprocess.run(
        [sys.executable, "script.py"],
        cwd=TEST_ROOT,
        text=True,
        capture_output=True,
        timeout=300,
    )

    print(result.stdout)
    print(result.stderr)

    assert result.returncode == 0, "script.py dry-run 실패"

    dry_submission = TEST_ROOT / "output" / "submission.csv"
    assert dry_submission.exists(), "submission.csv가 생성되지 않음"

    display(pd.read_csv(dry_submission))
    print("✅ dry-run 성공")
else:
    print("RUN_DRY_RUN=False — 필요할 때 True로 바꾸고 실행하세요.")

# 다음 단계: 실제 leaderboard 성능 개선

현재 notebook은 **Voice/AASIST starter**입니다.

DACON 점수에서 Music EER의 영향도도 크므로, 다음 순서로 확장하는 것을 권장합니다.

1. **AASIST + RawBoost / Codec / telephone augmentation**
2. ASVspoof2021 DF 등 speech deepfake 추가
3. **FakeMusicCaps** 등 music real/fake 추가
4. CtrSVDD 등 singing/voice 데이터 추가
5. Voice + Music 혼합 샘플 synthetic mixing
6. **WPT-XLSR-AASIST** backbone
7. 5-head multi-task:
   - File Fake
   - Voice Fake
   - Music Fake
   - Voice Presence
   - Music Presence
8. 4초 segment 추론 + top-k aggregation
9. AASIST / WPT-XLSR-AASIST / spectral CNN score ensemble

이번 `submit.zip`은 형식 검증과 첫 leaderboard 기준점을 만들기 위한 파일로 사용하는 것이 좋습니다.